## Imports

In [1]:
import numpy as np 
import pandas as pd 
import librosa
from transformers import AutoModelForAudioClassification,AutoFeatureExtractor
import os
import torch
from torch.utils.data import Dataset,DataLoader
import glob
import torch.nn as nn
from torch.optim.lr_scheduler import CosineAnnealingLR
import wandb
from kaggle_secrets import UserSecretsClient
import random

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
user_secrets = UserSecretsClient()
wandb_api = user_secrets.get_secret("login")
os.environ["WANDB_API_KEY"]=wandb_api
wandb.login()
print("Wandb login successful")


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: 23f1000559 (23f1000559-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Wandb login successful


## Model

In [3]:
model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
model = AutoModelForAudioClassification.from_pretrained(
    model_name,num_labels=10,ignore_mismatched_sizes=True
)
feature_extractor = AutoFeatureExtractor.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

2026-04-09 08:02:47.090676: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775721767.328584      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775721767.384849      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775721767.883041      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775721767.883080      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775721767.883083      55 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ASTForAudioClassification were not initialized from the model checkpoint at MIT/ast-finetuned-audioset-10-10-0.4593 and are newly initialized because the shapes did not match:
- classifier.dense.bias: found shape torch.Size([527]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.dense.weight: found shape torch.Size([527, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [4]:
print(model,feature_extractor)

ASTForAudioClassification(
  (audio_spectrogram_transformer): ASTModel(
    (embeddings): ASTEmbeddings(
      (patch_embeddings): ASTPatchEmbeddings(
        (projection): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ASTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ASTLayer(
          (attention): ASTAttention(
            (attention): ASTSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
            )
            (output): ASTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ASTIntermediate(
            (dense): Linear(in_features=768, out_features=3072, bias=T

In [5]:
# freeze all layers
for param in model.parameters():
    param.requires_grad=False
# unfreeze last 8 layers and classifier head
for layer in model.audio_spectrogram_transformer.encoder.layer[-8:]:
    for param in layer.parameters():
        param.requires_grad=True
for param in model.classifier.parameters():
    param.requires_grad=True


# Dataset and Loader

In [6]:
# all song directories and their labels
song_dirs = []
labels=[]
genres = ["blues","classical","country","disco","hiphop","jazz","metal","pop","reggae","rock"]
root_dir = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
for genre_idx, genre in enumerate(genres):
    genre_path = os.path.join(root_dir, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        song_dirs.append(song_path)
        labels.append(genre_idx)

In [7]:
# noise files
noise = glob.glob("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio/*.wav")

In [8]:
class MyDataset(Dataset):
    def __init__(self,song_dirs,labels,noise,feature_extractor,sr=16000,duration=10):
        super().__init__()
        self.song_dirs = song_dirs
        self.labels = labels
        self.noise = noise
        self.feature_extractor = feature_extractor
        self.sr = sr
        self.max_len = sr * duration 
        self.genre_to_dirs = {}
        for d , l in zip(song_dirs,labels):
            self.genre_to_dirs.setdefault(l,[]).append(d)
    def pad_truncate(self,audio):  
        if len(audio)> self.max_len:
            audio = audio[:self.max_len]
        else:
            audio = np.pad(audio,(0,self.max_len-len(audio)))
        return audio
    def add_noise(self,audio): 
        noise_file = random.choice(self.noise) 
        noise,_ = librosa.load(noise_file,sr=self.sr)
        noise = self.pad_truncate(noise)
        noise_level = np.random.uniform(0.1,0.5)
        return audio + noise * noise_level
    def load_stem(self,song_dir,stem):
        stem_path = os.path.join(song_dir,stem)
        stem_audio,_ = librosa.load(stem_path,sr = self.sr)
        return self.pad_truncate(stem_audio)
    def __len__(self):
        return len(self.song_dirs)
    def __getitem__(self,idx):
        label = self.labels[idx]
        stems =["bass.wav","vocals.wav","drums.wav","other.wav"]
        # diff song_dir for each stem
        same_genre_dirs = self.genre_to_dirs[label]
        stem_audios = []
        for stem in stems:
            song_dir = random.choice(same_genre_dirs)
            stem_audios.append(self.load_stem(song_dir,stem))
        # random weight mixing of stems
        weights = np.random.uniform(0.5,1.5,4)
        mix = sum(w*a for w,a in zip(weights,stem_audios))
        # tempo stretching
        if np.random.rand()<0.5:
            rate = np.random.uniform(0.85,1.15)
            mix = librosa.effects.time_stretch(mix, rate=rate)
            mix = self.pad_truncate(mix)
        # add noise
        mix = self.add_noise(mix)
        # normalise
        mix = mix/(np.max(np.abs(mix)) + 1e-6)
        inputs = self.feature_extractor(
            mix, sampling_rate=self.sr, return_tensors="pt"
        )
        return inputs["input_values"].squeeze(0), torch.tensor(label)
        

In [9]:
# dataset and loader instance
train_dataset = MyDataset(song_dirs,labels,noise,feature_extractor)
train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True)

## Training Loop

In [10]:
num_epochs = 15
device = "cuda" if torch.cuda.is_available() else "gpu"

In [11]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW([
    {"params": model.audio_spectrogram_transformer.encoder.layer[-8:].parameters(),"lr":1e-5},
    {"params": model.classifier.parameters(),"lr":1e-4}
])
scheduler = CosineAnnealingLR(optimizer,T_max = num_epochs,eta_min=1e-7)

In [12]:
# Start a new wandb run 
run = wandb.init(
    # Set the wandb project where this run will be logged.
    project="23f1000559-t12026",
    entity ="23f1000559-indian-institute-of-technology-madras",
    # Track hyperparameters and run metadata.
    config={
        "epochs": 15,
        "batch_size": 8,
        "learning_rate": [1e-4 ,1e-5],
        "model": "MIT/ast-finetuned-audioset-final"
    },
)

In [13]:
model.to(device)
for epoch in range(num_epochs):
    total_loss=0
    model.train()
    for x,y in train_loader:
        x = x.to(device)
        y = y.to(device)
        output = model(x)
        logits = output.logits
        loss = criterion(logits,y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()
    avg_loss = total_loss/len(train_loader)
    scheduler.step()
    #Log loss to wandb
    wandb.log({"epoch": epoch+1, "train_loss": avg_loss})
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

Epoch 1/15, Loss: 1.4454
Epoch 2/15, Loss: 0.9384
Epoch 3/15, Loss: 0.8634
Epoch 4/15, Loss: 0.8280
Epoch 5/15, Loss: 0.7625
Epoch 6/15, Loss: 0.7684
Epoch 7/15, Loss: 0.7123
Epoch 8/15, Loss: 0.7439
Epoch 9/15, Loss: 0.7269
Epoch 10/15, Loss: 0.6882
Epoch 11/15, Loss: 0.6767
Epoch 12/15, Loss: 0.6555
Epoch 13/15, Loss: 0.6616
Epoch 14/15, Loss: 0.6644
Epoch 15/15, Loss: 0.6519


In [14]:
#save model
path = "/kaggle/working/ast"
model.save_pretrained(path)
feature_extractor.save_pretrained(path)

['/kaggle/working/ast/preprocessor_config.json']

In [15]:
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret("huggingface")
# Login to HF
login(token=hf_token)

In [16]:
# model to huggingface
repo_name = "m7k-run/ast-genre-classifier"  

model.push_to_hub(repo_name)
feature_extractor.push_to_hub(repo_name)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/m7k-run/ast-genre-classifier/commit/76e857dfcc42fcd8fba3839fcfc86f29b346072e', commit_message='Upload feature extractor', commit_description='', oid='76e857dfcc42fcd8fba3839fcfc86f29b346072e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/m7k-run/ast-genre-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='m7k-run/ast-genre-classifier'), pr_revision=None, pr_num=None)

## Submission

In [17]:
genres = ["blues","classical","country","disco","hiphop",
          "jazz","metal","pop","reggae","rock"]
idx_genre = {i: g for i , g in enumerate(genres)}

In [18]:
test_df = pd.read_csv("/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/test.csv")

In [19]:
def pad_truncate(audio):
        max_len = 160000
        if len(audio) > max_len:
            audio = audio[:max_len]
        else: 
            audio = np.pad(audio,(0,(max_len-len(audio))))
        return audio

In [20]:
def predict_genre_tta(file_path, model, feature_extractor, device, n_crops=5):
    audio, sr = librosa.load(file_path, sr=16000)
    crop_len = 160000  
    min_len = 400     

    if len(audio) < 16000:
        audio = np.pad(audio, (0, 16000 - len(audio)))

    all_logits = []
    for _ in range(n_crops):
        if len(audio) >= crop_len:
            start = random.randint(0, len(audio) - crop_len)
            crop = audio[start:start + crop_len]
        else:
            crop = pad_truncate(audio)
        
        crop = crop / (np.max(np.abs(crop)) + 1e-6)
        inputs = feature_extractor([crop], sampling_rate=16000, return_tensors="pt")
        with torch.no_grad():
            logits = model(inputs["input_values"].to(device)).logits
        all_logits.append(logits)

    avg_logits = torch.stack(all_logits).mean(0)
    return idx_genre[avg_logits.argmax(dim=1).item()]

In [21]:
model.eval()
model.to(device)
predictions = []
for file_path in test_df["filename"]:
    root = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
    final_path = os.path.join(root, file_path)
    genre_pred = predict_genre_tta(final_path, model, feature_extractor, device)
    predictions.append(genre_pred)

In [22]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": predictions
})
submission.to_csv("submission.csv", index=False)
print(submission.head())

   id      genre
0   1        pop
1   2  classical
2   3      disco
3   4      metal
4   5    country
